In [1]:
import umap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px

from sklearn.decomposition import PCA
from scipy.signal import savgol_filter

## Read csv

In [4]:
df = pd.read_excel('../../Jataí_infravermelho_duas_coletas.xlsx')
df = df.T.reset_index()

columns = df.iloc[0].tolist()
columns[0:3] = ['Dia_coleta', 'Abelha', 'IDX_produtor']

df.columns = columns
df = df[1:]

df['Rodada_coleta'] = df['Dia_coleta'].str.extract(r'(\d+)º').astype(int).values
df['IDX_produtor'] = df['IDX_produtor'].str.replace('Produtor ', '', regex=False).astype(int)
df['Dia_coleta'] = df.groupby('Rodada_coleta').cumcount() + 1

df.head()

,Dia_coleta,Abelha,IDX_produtor,4000,3999,3998,3997,3996,3995,3994,...,658,657,656,655,654,653,652,651,650,Rodada_coleta
1,1,JATAÍ,1,100,99.995513,99.992737,99.9917,99.99185,99.992837,99.994127,...,99.322983,98.514537,98.14798,98.87685,98.15767,96.38628,97.676523,98.939357,100.83396,1
2,2,JATAÍ,2,100,99.99462,99.994017,99.988213,99.982753,99.98002,99.981893,...,99.101097,98.686977,99.0192,99.381357,98.198847,97.36968,97.516307,99.32453,100.516735,1
3,3,JATAÍ,3,100,100.001873,100.003493,100.003477,100.00228,100.00145,100.0017,...,98.023333,97.086957,96.893953,97.247947,97.3147,97.140197,97.92563,99.290683,101.011927,1
4,4,JATAÍ,4,100,100.001053,100.000393,99.999377,99.994727,99.994683,99.997733,...,100.155547,99.407333,99.56157,98.271427,96.879717,96.023513,97.0668,98.72557,100.74969,1
5,5,JATAÍ,5,100,100.002107,100.002913,100.003307,99.99521,99.997057,100.00043,...,100.36313,99.592973,99.704393,100.024977,96.684607,95.810703,96.893257,98.506587,100.467617,1


### [Linha temporal]

In [ ]:
metadata_cols = ['Dia_coleta', 'Abelha', 'IDX_produtor', 'Rodada_coleta']
freq_cols     = [col for col in df.columns if col not in metadata_cols]

df_plot = df.melt(
    id_vars=metadata_cols, 
    value_vars=freq_cols, 
    var_name='Frequencia_cm1', 
    value_name='Intensidade'
)

df_plot['Frequencia_cm1'] = df_plot['Frequencia_cm1'].astype(float)

df_plot['Legenda'] = 'Produtor ' + df_plot['IDX_produtor'].astype(str)

df_plot['Rodada_coleta'] = df_plot['Rodada_coleta'].map({1: '1º Dia', 2: '2º Dia'})

fig = px.line(
    df_plot,
    x='Frequencia_cm1',
    y='Intensidade',
    color='Legenda',        
    line_dash='Rodada_coleta', 
    hover_name='Abelha',
    title='Espectro de Frequência das Amostras de Mel (FTIR)',
    labels={
        'Frequencia_cm1': 'Número de Onda (cm⁻¹)',
        'Intensidade': 'Intensidade / Absorbância',
        'Rodada_coleta': 'Coleta' 
    },
    template='plotly_white'
)

fig.update_layout(
    xaxis=dict(autorange='reversed'),
    legend_title_text='Amostras',
    hovermode="x unified" 
)

fig.update_traces(line=dict(width=1.5))

fig.update_layout(
    width=1000,  
    height=700, 
)
fig.write_html("graficos/Jatai_Frequencias.html")

### [PCA]

In [ ]:
pca  = PCA(n_components=2)
data = df.drop(columns=['Dia_coleta', 'Abelha', 'IDX_produtor', 'Rodada_coleta']).values

pca_df = pd.DataFrame(
    data=pca.fit_transform(data), 
    columns=['PC1', 'PC2']
)

pca_df['IDX_produtor'] = ('Produtor ' + df['IDX_produtor'].astype(str)).values
pca_df['Rodada_coleta'] = (df['Rodada_coleta'].astype(str) + 'ª Coleta').values
pca_df['Dia_coleta'] = ('Dia ' + df['Dia_coleta'].astype(str)).values
pca_df['Abelha'] = df['Abelha'].values

fig = px.scatter(
    pca_df, 
    x='PC1', 
    y='PC2',
    color='IDX_produtor',  
    text='Rodada_coleta',  
    hover_name='Abelha',      
    hover_data={
        'IDX_produtor': False, 
        'Rodada_coleta': True,
        'Dia_coleta': True,
        'PC1': False,  
        'PC2': False
    },
    title='Análise PCA: Agrupamento de Produtores e Coletas',
    labels={
        'PC1': f'Componente Principal 1 ({pca.explained_variance_ratio_[0]*100:.1f}%)',
        'PC2': f'Componente Principal 2 ({pca.explained_variance_ratio_[1]*100:.1f}%)'
    },
    template='plotly_white'   
)


fig.update_traces(
    textposition='top center',
    textfont_size=9, 
    marker=dict(size=10, opacity=0.8, line=dict(width=1, color='DarkSlateGrey'))
)

fig.update_layout(
    width=700,  
    height=700, 
    yaxis=dict(
        scaleanchor="x", 
        scaleratio=1   
    )
)
fig.write_html("graficos/Jatai_PCA_2D_Todas.html")

In [ ]:
cols_metadados = ['Dia_coleta', 'Abelha', 'IDX_produtor', 'Rodada_coleta']
data = df.drop(columns=cols_metadados).values

pca = PCA(n_components=3, svd_solver='randomized', random_state=42)
pca_df = pd.DataFrame(
    data=pca.fit_transform(data), 
    columns=['PC1', 'PC2', 'PC3'] # Adicionando a 3ª coluna
)

pca_df['IDX_produtor'] = ('Produtor ' + df['IDX_produtor'].astype(str)).values
pca_df['Rodada_coleta'] = (df['Rodada_coleta'].astype(str) + 'ª Coleta').values
pca_df['Dia_coleta'] = ('Dia ' + df['Dia_coleta'].astype(str)).values
pca_df['Abelha'] = df['Abelha'].values

fig_pca = px.scatter_3d(
    pca_df, 
    x='PC1', 
    y='PC2',
    z='PC3', # Adicionando o eixo Z
    color='IDX_produtor',  
    text='Rodada_coleta',  
    hover_name='Abelha',      
    hover_data={'IDX_produtor': False, 'Rodada_coleta': True, 'Dia_coleta': True, 'PC1': False, 'PC2': False, 'PC3': False},
    title='Análise PCA 3D',
    labels={
        'PC1': f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)',
        'PC2': f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)',
        'PC3': f'PC3 ({pca.explained_variance_ratio_[2]*100:.1f}%)'
    },
    template='plotly_white'   
)

# 5. Ajustes visuais 3D
fig_pca.update_traces(
    textposition='top center',
    textfont_size=8, 
    marker=dict(size=6, opacity=0.8, line=dict(width=1, color='DarkSlateGrey')) # Tamanho do marcador menor para 3D
)

fig_pca.update_layout(width=800, height=800)
fig.write_html("graficos/Jatai_PCA_3D_Todas.html")

#### [Faixas específicas]

In [ ]:
freqs = [(3800, 3015), (3015, 2450), (1770, 1530), (1520, 1200), (1200, 905), (905, 700)]

for freq_max, freq_min in freqs:
    cols_metadados = ['Dia_coleta', 'Abelha', 'IDX_produtor', 'Rodada_coleta']
    todas_freqs = [col for col in df.columns if col not in cols_metadados]

    freqs_selecionadas = [
        col for col in todas_freqs 
        if freq_min <= float(col) <= freq_max
    ]


    data_filtrado = df[freqs_selecionadas].values

    pca = PCA(n_components=2)
    pca_df = pd.DataFrame(
        data=pca.fit_transform(data_filtrado), 
        columns=['PC1', 'PC2']
    )

    pca_df['IDX_produtor'] = ('Produtor ' + df['IDX_produtor'].astype(str)).values
    pca_df['Rodada_coleta'] = (df['Rodada_coleta'].astype(str) + 'ª Coleta').values
    pca_df['Abelha'] = df['Abelha'].values

    fig = px.scatter(
        pca_df, 
        x='PC1', 
        y='PC2',
        color='IDX_produtor',  
        text='Rodada_coleta',  
        hover_name='Abelha',      
        hover_data={
            'IDX_produtor': False, 
            'Rodada_coleta': True,
            'PC1': False,  
            'PC2': False
        },
        title=f'Análise PCA (Frequências Restritas: {freq_max} a {freq_min} cm⁻¹)',
        labels={
            'PC1': f'Componente Principal 1 ({pca.explained_variance_ratio_[0]*100:.1f}%)',
            'PC2': f'Componente Principal 2 ({pca.explained_variance_ratio_[1]*100:.1f}%)'
        },
        template='plotly_white'   
    )

    fig.update_traces(
        textposition='top center',
        textfont_size=9, 
        marker=dict(size=10, opacity=0.8, line=dict(width=1, color='DarkSlateGrey'))
    )

    fig.update_layout(
        width=700,  
        height=700, 
        yaxis=dict(
            scaleanchor="x", 
            scaleratio=1   
        )
    )
    fig.write_html(f"graficos/Jatai_PCA_2D_{freq_max}_{freq_min}.html")

In [ ]:
freqs = [(3800, 3015), (3015, 2450), (1770, 1530), (1520, 1200), (1200, 905), (905, 700)]

for freq_max, freq_min in freqs:
    cols_metadados = ['Dia_coleta', 'Abelha', 'IDX_produtor', 'Rodada_coleta']
    todas_freqs = [col for col in df.columns if col not in cols_metadados]

    freqs_selecionadas = [
        col for col in todas_freqs 
        if freq_min <= float(col) <= freq_max
    ]

    data_filtrado = df[freqs_selecionadas].values

    # MUDANÇA: n_components para 3
    pca = PCA(n_components=3)
    pca_df = pd.DataFrame(
        data=pca.fit_transform(data_filtrado), 
        columns=['PC1', 'PC2', 'PC3'] # MUDANÇA: Adicionado PC3
    )

    pca_df['IDX_produtor'] = ('Produtor ' + df['IDX_produtor'].astype(str)).values
    pca_df['Rodada_coleta'] = (df['Rodada_coleta'].astype(str) + 'ª Coleta').values
    pca_df['Abelha'] = df['Abelha'].values

    # MUDANÇA: Usando scatter_3d
    fig = px.scatter_3d(
        pca_df, 
        x='PC1', 
        y='PC2',
        z='PC3', # MUDANÇA: Parâmetro z adicionado
        color='IDX_produtor',  
        text='Rodada_coleta',  
        hover_name='Abelha',      
        hover_data={
            'IDX_produtor': False, 
            'Rodada_coleta': True,
            'PC1': False,  
            'PC2': False,
            'PC3': False # MUDANÇA: Ocultando PC3 no hover para manter o padrão
        },
        title=f'Análise PCA 3D (Frequências Restritas: {freq_max} a {freq_min} cm⁻¹)',
        labels={
            'PC1': f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)',
            'PC2': f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)',
            'PC3': f'PC3 ({pca.explained_variance_ratio_[2]*100:.1f}%)' # MUDANÇA: Label do PC3
        },
        template='plotly_white'   
    )

    fig.update_traces(
        textposition='top center',
        textfont_size=9, 
        # Diminuí o tamanho (size) de 10 para 6, pois em 3D as bolhas tendem a se sobrepor muito
        marker=dict(size=6, opacity=0.8, line=dict(width=1, color='DarkSlateGrey')) 
    )

    fig.update_layout(
        width=800,  # Aumentei levemente a largura para acomodar melhor a rotação 3D
        height=800, 
        # Removi o bloqueio de proporção do yaxis, pois o Plotly 3D gerencia isso nativamente
    )
    
    # MUDANÇA: Nome do arquivo salvo atualizado para 3D
    fig.write_html(f"graficos/Jatai_PCA_3D_{freq_max}_{freq_min}.html")

### [UMAP]

In [ ]:
import os
os.makedirs('graficos/', exist_ok=True)

freqs = [
    (3800, 3015), (3015, 2450), (1770, 1530), 
    (1520, 1200), (1200, 905), (905, 700), 
    ('Todas', 'Todas')
]

vizinhos_list = [3, 4, 5]
metricas_list = ['cosine', 'euclidean']
min_dist_list = [0.1, 0.8]

total_graficos = len(freqs) * len(vizinhos_list) * len(metricas_list) * len(min_dist_list)
contador = 1

for freq_max, freq_min in freqs:
    cols_metadados = ['Dia_coleta', 'Abelha', 'IDX_produtor', 'Rodada_coleta']
    todas_freqs = [col for col in df.columns if col not in cols_metadados]

    if freq_max == 'Todas':
        freqs_selecionadas = todas_freqs
        texto_titulo = "Todas as Frequências"
        texto_arquivo = "Todas"
    else:
        freqs_selecionadas = [
            col for col in todas_freqs 
            if freq_min <= float(col) <= freq_max
        ]
        texto_titulo = f"{freq_max} a {freq_min} cm⁻¹"
        texto_arquivo = f"{freq_max}_{freq_min}"

    data_filtrado = df[freqs_selecionadas].values

    for nn in vizinhos_list:
        for metrica in metricas_list:
            for md in min_dist_list:
                
                print(f"Gerando gráfico {contador}/{total_graficos} | Freq: {texto_arquivo} | N={nn} | Metric={metrica} | MinDist={md}")
                
                # Instancia e treina o UMAP (2 componentes para ser 2D)
                redutor = umap.UMAP(
                    n_neighbors=nn,
                    n_components=2, 
                    metric=metrica,
                    min_dist=md,
                    random_state=42
                )
                
                umap_df = pd.DataFrame(
                    data=redutor.fit_transform(data_filtrado), 
                    columns=['UMAP1', 'UMAP2']
                )

                umap_df['IDX_produtor'] = ('Produtor ' + df['IDX_produtor'].astype(str)).values
                umap_df['Rodada_coleta'] = (df['Rodada_coleta'].astype(str) + 'ª Coleta').values
                umap_df['Abelha'] = df['Abelha'].values

                fig = px.scatter(
                    umap_df, 
                    x='UMAP1', 
                    y='UMAP2',
                    color='IDX_produtor',  
                    text='Rodada_coleta',  
                    hover_name='Abelha',      
                    title=f'UMAP 2D ({texto_titulo}) | vizinhos:{nn} | dist:{md} | {metrica}',
                    template='plotly_white'   
                )

                fig.update_traces(
                    textposition='top center',
                    textfont_size=9, 
                    marker=dict(size=10, opacity=0.8, line=dict(width=1, color='DarkSlateGrey'))
                )

                fig.update_layout(
                    width=700, 
                    height=700,
                    yaxis=dict(
                        scaleanchor="x", 
                        scaleratio=1   
                    )
                )
                
                nome_arquivo = f"graficos/Jatai_UMAP_2D_{texto_arquivo}_{metrica}_nn{nn}_md{md}.html"
                
                fig.write_html(nome_arquivo)
                
                contador += 1

print("\nConcluído! Verifique a pasta 'graficos/'")

In [ ]:
import umap
import plotly.express as px
import pandas as pd
import os

os.makedirs('graficos/', exist_ok=True)

freqs = [
    (3800, 3015), (3015, 2450), (1770, 1530), 
    (1520, 1200), (1200, 905), (905, 700), 
    ('Todas', 'Todas')
]

vizinhos_list = [3, 4, 5]
metricas_list = ['cosine', 'euclidean']
min_dist_list = [0.1, 0.8]

total_graficos = len(freqs) * len(vizinhos_list) * len(metricas_list) * len(min_dist_list)
contador = 1

for freq_max, freq_min in freqs:
    cols_metadados = ['Dia_coleta', 'Abelha', 'IDX_produtor', 'Rodada_coleta']
    todas_freqs = [col for col in df.columns if col not in cols_metadados]

    if freq_max == 'Todas':
        freqs_selecionadas = todas_freqs
        texto_titulo = "Todas as Frequências"
        texto_arquivo = "Todas"
    else:
        freqs_selecionadas = [
            col for col in todas_freqs 
            if freq_min <= float(col) <= freq_max
        ]
        texto_titulo = f"{freq_max} a {freq_min} cm⁻¹"
        texto_arquivo = f"{freq_max}_{freq_min}"

    data_filtrado = df[freqs_selecionadas].values

    for nn in vizinhos_list:
        for metrica in metricas_list:
            for md in min_dist_list:
                
                print(f"Gerando gráfico {contador}/{total_graficos} | Freq: {texto_arquivo} | N={nn} | Metric={metrica} | MinDist={md}")
                
                # MUDANÇA: n_components alterado para 3
                redutor = umap.UMAP(
                    n_neighbors=nn,
                    n_components=3, 
                    metric=metrica,
                    min_dist=md,
                    random_state=42
                )
                
                # MUDANÇA: Adicionado UMAP3
                umap_df = pd.DataFrame(
                    data=redutor.fit_transform(data_filtrado), 
                    columns=['UMAP1', 'UMAP2', 'UMAP3']
                )

                umap_df['IDX_produtor'] = ('Produtor ' + df['IDX_produtor'].astype(str)).values
                umap_df['Rodada_coleta'] = (df['Rodada_coleta'].astype(str) + 'ª Coleta').values
                umap_df['Abelha'] = df['Abelha'].values

                # MUDANÇA: px.scatter_3d e eixo z incluído
                fig = px.scatter_3d(
                    umap_df, 
                    x='UMAP1', 
                    y='UMAP2',
                    z='UMAP3',
                    color='IDX_produtor',  
                    text='Rodada_coleta',  
                    hover_name='Abelha',      
                    title=f'UMAP 3D ({texto_titulo}) | vizinhos:{nn} | dist:{md} | {metrica}',
                    template='plotly_white'   
                )

                # MUDANÇA: Tamanho do marcador reduzido para 6
                fig.update_traces(
                    textposition='top center',
                    textfont_size=9, 
                    marker=dict(size=6, opacity=0.8, line=dict(width=1, color='DarkSlateGrey'))
                )

                # MUDANÇA: yaxis removido (o 3D gerencia isso nativamente) e tamanho da tela levemente aumentado
                fig.update_layout(
                    width=800, 
                    height=800
                )
                
                # MUDANÇA: Nome do arquivo ajustado para 3D
                nome_arquivo = f"graficos/Jatai_UMAP_3D_{texto_arquivo}_{metrica}_nn{nn}_md{md}.html"
                
                fig.write_html(nome_arquivo)
                
                contador += 1

print("\nConcluído! Verifique a pasta 'graficos/'")